# 02 — Concept Bottleneck Training  ·  Critical Fix (M13 → strict bottleneck) + Gate G3 inputs

Trains the **strict concept bottleneck** and its controls, producing the three numbers **Gate G3** reads to pick
Path A vs Path B:
- **tradeoff** = opaque − independent disease accuracy (the interpretability *cost*),
- **leakage proxy** = leaky − independent (how much the bottleneck can be bypassed),
- per-class disease F1 (watch for **COPD collapse** — the §2.5 validity-hole failure mode).

Variants: `independent` (disease from concepts only — the true bottleneck), `sequential` (features→concepts→disease),
`leaky` (concepts + features — the cheat control), `opaque` (features only — the accuracy upper bound).

**Requires:** `concepts_all.npz` from notebook 01, and **frozen M2 features** aligned to the same cycle index
(fill in `get_M2_features`). Outputs: `results_M13cbm_<variant>.json` (§4 schema, with confusion matrix, CIs,
per-patient score dumps) + a tradeoff summary.


In [ ]:
import sys, os
OWMTL_PKG = ".."
sys.path.insert(0, OWMTL_PKG)
import numpy as np, json, torch
from collections import defaultdict
from owmtl.bottleneck import ConceptBottleneck, MODES
from owmtl import eval_utils as E
from owmtl.icbhi_data import KNOWN_DISEASES
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "/kaggle/working"; SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)


In [ ]:
# --- load concepts from notebook 01 ------------------------------------------
Z = np.load(os.path.join(OUT_DIR, "concepts_all.npz"), allow_pickle=True)
X = Z["X"].astype("float32")                 # (N_cycles, N_concepts)
patient = Z["patient"]; split = Z["split"]; diagnosis = Z["diagnosis"]
concept_names = list(Z["concept_names"])
print("concept matrix", X.shape)


In [ ]:
# --- M2 frozen features aligned to the SAME cycle order ----------------------
# TODO: return an (N_cycles, feature_dim) array from your frozen M2 encoder, in the
# exact order of X (i.e. the record order from notebook 01). Easiest: in nb01, also
# run each cycle's mel-spectrogram through frozen M2 and save M2_features.npy.
def get_M2_features():
    path = os.path.join(OUT_DIR, "M2_features.npy")
    if os.path.exists(path):
        return np.load(path).astype("float32")
    raise FileNotFoundError(
        "M2_features.npy not found. independent-CBM can run on concepts alone, but "
        "sequential/leaky/opaque need frozen M2 features. Export them aligned to the "
        "cycle index (same order as concepts_all.npz).")

try:
    F = get_M2_features(); HAVE_FEATS = True
except Exception as ex:
    print("NOTE:", ex); F = np.zeros((len(X), 1), dtype="float32"); HAVE_FEATS = False
FEATURE_DIM = F.shape[1]


In [ ]:
# --- patient-level aggregation (disease is a patient-level label) ------------
# Known-class disease task (COPD/Healthy/URTI). Aggregate a patient's cycles by mean.
lab_map = {d: i for i, d in enumerate(KNOWN_DISEASES)}
keep = np.array([d in lab_map for d in diagnosis])
def agg(patients_mask):
    by = defaultdict(list)
    for i in np.where(patients_mask)[0]:
        by[patient[i]].append(i)
    pids, Xc, Ff, yy, sp = [], [], [], [], []
    for pid, idxs in by.items():
        idxs = np.array(idxs)
        pids.append(pid)
        Xc.append(X[idxs].mean(0)); Ff.append(F[idxs].mean(0))
        yy.append(lab_map[diagnosis[idxs[0]]]); sp.append(split[idxs[0]])
    return (np.array(pids), np.stack(Xc), np.stack(Ff), np.array(yy), np.array(sp))

pids, Xp, Fp, yp, spp = agg(keep)
tr, te = spp == "train", spp == "test"
print(f"patients: {len(pids)}  train {tr.sum()}  test {te.sum()}  classes {KNOWN_DISEASES}")


In [ ]:
# --- train one variant, return metrics + per-patient predictions -------------
def run_variant(mode, epochs=150, lr=1e-3, hidden=64):
    Cdim, Fdim, K = Xp.shape[1], FEATURE_DIM, len(KNOWN_DISEASES)
    model = ConceptBottleneck(Cdim, Fdim, K, mode=mode, hidden=hidden).to(DEVICE)
    xc = torch.tensor(Xp).to(DEVICE); xf = torch.tensor(Fp).to(DEVICE)
    y = torch.tensor(yp).long().to(DEVICE)
    # class weights for imbalance (URTI is small)
    cw = torch.tensor([ (yp[tr]==k).sum() for k in range(K) ], dtype=torch.float)
    cw = (cw.sum()/(cw+1e-6)); cw = (cw/cw.sum()*K).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    tr_t = torch.tensor(tr);
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        logits, chat = model(xf[tr_t], xc[tr_t])
        loss = model.loss(logits, y[tr_t], chat, xc[tr_t], class_weight=cw)
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        logits, _ = model(xf, xc)
        proba = torch.softmax(logits, -1).cpu().numpy()
        pred = proba.argmax(1)
    return model, pred, proba


In [ ]:
# --- run all variants, assemble the G3 tradeoff -----------------------------
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
results = {}
for mode in MODES:
    if mode != "independent" and not HAVE_FEATS:
        print(f"skip {mode}: needs M2 features"); continue
    model, pred, proba = run_variant(mode)
    acc = accuracy_score(yp[te], pred[te])
    f1m = f1_score(yp[te], pred[te], average="macro")
    perclass_f1 = f1_score(yp[te], pred[te], average=None, labels=list(range(len(KNOWN_DISEASES))))
    cm = confusion_matrix(yp[te], pred[te], labels=list(range(len(KNOWN_DISEASES))))
    # bootstrap CI on macro-F1
    f1_ci = E.bootstrap_ci(lambda a,b: f1_score(a,b,average="macro",labels=list(range(len(KNOWN_DISEASES)))),
                           yp[te], pred[te], n_boot=1000, seed=SEED)
    # dump per-patient scores (max known-class prob as the confidence)
    E.dump_scores(os.path.join(OUT_DIR, f"M13cbm_{mode}"),
                  ids=pids[te], scores=proba[te].max(1), labels=yp[te])
    results[mode] = {"acc": float(acc), "macro_f1": float(f1m),
                     "macro_f1_ci95": [round(f1_ci[1],4), round(f1_ci[2],4)],
                     "per_class_f1": {KNOWN_DISEASES[i]: float(perclass_f1[i]) for i in range(len(KNOWN_DISEASES))},
                     "confusion_matrix_raw": cm.tolist(), "pred": pred}
    print(f"{mode:11s}: acc {acc:.3f}  macroF1 {f1m:.3f}  CI[{f1_ci[1]:.3f},{f1_ci[2]:.3f}]  "
          f"perclass {dict(zip(KNOWN_DISEASES, np.round(perclass_f1,3)))}")


In [ ]:
# --- Gate G3 inputs: tradeoff, leakage proxy, COPD-collapse check ------------
g3 = {}
if "independent" in results and "opaque" in results:
    g3["interpretability_cost_acc"] = round(results["opaque"]["acc"] - results["independent"]["acc"], 4)
    g3["interpretability_cost_f1"]  = round(results["opaque"]["macro_f1"] - results["independent"]["macro_f1"], 4)
if "independent" in results and "leaky" in results:
    g3["leakage_proxy_acc"] = round(results["leaky"]["acc"] - results["independent"]["acc"], 4)
# COPD-collapse: does independent just predict COPD?
if "independent" in results:
    ind_pred = results["independent"]["pred"][te]
    g3["independent_predicts_COPD_frac"] = round(float(np.mean(ind_pred == 0)), 3)
    g3["independent_COPD_f1"] = results["independent"]["per_class_f1"]["COPD"]
print("GATE G3 INPUTS:", json.dumps(g3, indent=2))


In [ ]:
# --- write §4-schema results JSON per variant -------------------------------
for mode, r in results.items():
    E.write_results_json(
        os.path.join(OUT_DIR, f"results_M13cbm_{mode}.json"),
        model_id=f"M13cbm_{mode}", model_name=f"Concept bottleneck ({mode})",
        contributor="Barshon", split="patient_independent_official_60_40",
        best_metrics={"accuracy": r["acc"], "f1_macro": r["macro_f1"],
                      "f1_macro_ci95": r["macro_f1_ci95"],
                      "per_class_f1": r["per_class_f1"],
                      "confusion_matrix_raw": r["confusion_matrix_raw"]},
        ablation={"ablation_group": "bottleneck_type", "ablation_role":
                  "baseline" if mode=="independent" else "variant",
                  "variable_changed": f"bottleneck_type: {mode}",
                  "component_flags": {"has_concept_bottleneck": mode in ("independent","sequential"),
                                      "bottleneck_type": mode,
                                      "concept_source": "physics"}},
        concept_metrics={"gate_G3": g3, "concept_names": concept_names},
        notes="Critical-Fix concept bottleneck; disease = f(concepts). Path A/B decided at G3.")
print("wrote results_M13cbm_*.json for:", list(results))


### G3 decision (read the printed `GATE G3 INPUTS`)
- **Path A (constructive)** if `independent` holds accuracy near `opaque` (small `interpretability_cost`),
  `leakage_proxy` is small, and it does **not** collapse to COPD (`independent_predicts_COPD_frac` not ≈1).
- **Path B (audit)** if the bottleneck costs a lot / leaks heavily / collapses toward COPD — that collapse is the
  *finding*, not a failure. Either way you have committed confusion matrices, CIs, and per-patient score dumps.
